# Homework 12

https://scikit-learn.org/0.15/modules/scaling_strategies.html#incremental-learning

* Implement a mini batch functionality to train a regressor.
    - (Optional) If anyone want to do this in a pipeline can do this: https://koaning.github.io/tokenwiser/api/pipeline.html

* Save model, load the model again and test it on `X_test` __Do NOT commit the pickle file__

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
def test_df():
    df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/car_prices/car_prices.csv', low_memory=False)

    df = df.sample(5000, random_state=100).reset_index(drop=True)
    
    y = df['sellingprice']
    df.drop('sellingprice', axis=1, inplace=True)
    X = df
    
    return X,y

def partial_df():
    df = pd.read_csv('https://raw.githubusercontent.com/msaricaumbc/DS_data/master/ds602/car_prices/car_prices.csv', low_memory=False)
   
    while(True):
        yield df.sample(100).reset_index(drop=True)
        
gen = partial_df()

In [3]:
X_test, y_test = test_df()

In [4]:
# each time you call this you will get a new slice of the dataframe.
next(gen)

,year,make,model,trim,body,transmission,vin,state,condition,odometer,color,interior,seller,mmr,sellingprice,saledate
0,2010,BMW,7 Series,750Li,sedan,NaN,wbakb8c58acy64266,pa,3.3,58533.0,gray,gray,towne bmw mini maserati of buffalo,29300,28700,Thu May 28 2015 02:30:00 GMT-0700 (PDT)
1,2013,Mercedes-Benz,C-Class,C250,Coupe,automatic,wddgj4hb4df971231,nv,3.6,14967.0,white,black,mercedes-benz financial services,25100,24250,Thu Jan 22 2015 06:00:00 GMT-0800 (PST)
2,2012,Infiniti,EX,EX35,SUV,automatic,jn1aj0hr6cm451504,il,3.9,46106.0,gray,black,nissan infiniti lt,22700,23500,Thu Feb 26 2015 02:00:00 GMT-0800 (PST)
3,2014,Jeep,Cherokee,Latitude,SUV,automatic,1c4pjmcs1ew232420,pa,4,29156.0,silver,black,pv holidng inc/non pgm,20100,20100,Thu May 28 2015 02:30:00 GMT-0700 (PDT)
4,2005,Dodge,Ram Pickup 1500,SLT,quad cab,automatic,1d7ha18d75j503061,ga,3.2,169141.0,red,gray,santander consumer,5500,8300,Tue Jun 16 2015 02:30:00 GMT-0700 (PDT)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2005,Chrysler,Town and Country,Touring,Minivan,automatic,2c4gp54l85r511724,nc,2.8,127835.0,silver,gray,santander consumer,2750,3600,Mon Mar 09 2015 02:30:00 GMT-0700 (PDT)
96,2013,Mitsubishi,Lancer,ES,Sedan,automatic,ja32u2fu8du020089,tn,2.8,41953.0,black,black,avis corporation,9825,9300,Wed Feb 25 2015 02:30:00 GMT-0800 (PST)
97,2013,Volkswagen,CC,R-Line PZEV,Sedan,automatic,wvwbp7an4de516491,ca,1.9,35276.0,gray,black,vw credit,16650,15250,Thu Jan 15 2015 04:30:00 GMT-0800 (PST)
98,2003,Kia,Sedona,EX,Minivan,NaN,kndup131x36482629,ga,1.9,95009.0,burgundy,gray,gwinnett place honda,1200,2200,Tue May 26 2015 06:00:00 GMT-0700 (PDT)


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import SGDRegressor
import pickle

model = SGDRegressor()
# Define pipeline steps
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

# Define column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, ['year', 'odometer', 'mmr', "year_of_sale"]),
        ('cat', categorical_transformer, ['make', 'model', 'trim', 'body', 'transmission', 'state', 'condition', 'color', 'interior'])
    ])

# Define the complete pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
])


In [6]:
def preprocess_data(df):
    df['mmr'] = pd.to_numeric(df['mmr'], errors='coerce')
    df["year_of_sale"] = df['saledate'].str.split().str[3].astype(float)
    return df

In [ ]:
from sklearn.metrics import mean_squared_error


def train_model(gen, pipeline, n_batches):
    first_batch = True
    for i in range(n_batches):
        
        batch = next(gen)
        X_batch = batch.drop('sellingprice', axis=1)
        X_batch = preprocess_data(X_batch)
        y_batch = batch['sellingprice']
        
        if i % 2000 == 1:
            pred = pipeline.predict(X_batch)
            print("iter : ", i, " mse : ", mean_squared_error(pred, y_batch))        
        if first_batch:
            pipeline.fit(X_batch, y_batch)
            first_batch = False
        else:
            pipeline.named_steps['regressor'].partial_fit(
                pipeline.named_steps['preprocessor'].transform(X_batch), 
                y_batch
            )
    return pipeline

n_batches = 20000  
trained_pipeline = train_model(gen, pipeline, n_batches)


/opt/anaconda3/lib/python3.9/site-packages/sklearn/linear_model/_stochastic_gradient.py:1503: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


iter :  1  mse :  5710965.2812507795
iter :  2001  mse :  2302130.453878935
iter :  4001  mse :  4648915.456496605
iter :  6001  mse :  1215191.6022588175
iter :  8001  mse :  1101286.3814710993
iter :  10001  mse :  4752832.4465678325


In [ ]:
# Save the model
with open('model.pkl', 'wb') as file:
    pickle.dump(trained_pipeline, file)

# Load the model
with open('model.pkl', 'rb') as file:
    loaded_pipeline = pickle.load(file)


In [ ]:
# Test the model
X_test_processed = preprocess_data(X_test)
predictions = loaded_pipeline.predict(X_test_processed)

# Calculate performance metrics
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, predictions)
print(f"Mean Squared Error: {mse}")
